In [ ]:
pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 67.2 MB/s eta 0:00:00


In [ ]:
!pip install torch transformers faiss-cpu tqdm requests openpyxl tiktoken


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 107.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 84.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 92.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 62.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstallin

In [ ]:
import pandas as pd
# Load data
afdb = pd.read_excel("/content/AFDB_New_Embe_V2.xlsx", engine="openpyxl")[["Filename", "Project Components", "Sector"]]
aiib = pd.read_excel("/content/AIIB_V2_Embd.xlsx", engine="openpyxl")[["Filename", "Project Components", "Sector"]]
world_bank = pd.read_excel("/content/WBG_New-Embeddings_Data_V2.xlsx", engine="openpyxl")[["Project ID", "Project Components", "Sector"]]

# Define processing columns
aiib_columns = ["Project Components"]
afdb_columns = ["Project Components"]
world_bank_columns = ["Project Components"]

In [ ]:
import logging
import numpy as np
import pandas as pd
from tqdm import tqdm
import faiss
import json
import torch
import gc  # For garbage collection
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from torch import Tensor
import time  # For timing execution

def average_pool(last_hidden_states: Tensor, attention_mask: Tensor) -> Tensor:
    last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)
    return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# Model Configuration
MODEL_NAME = "intfloat/multilingual-e5-large-instruct"
CHUNK_SIZE = 400  # Max token length supported
OVERLAP = 70
BATCH_SIZE = 32  # Micro-batching to reduce memory load
EMBEDDING_DIM = 1024  # Updated embedding dimension

# Initialize FAISS index
n_clusters = 14  # Match the number of sectors
faiss_index = faiss.IndexIVFFlat(
    faiss.IndexFlatL2(EMBEDDING_DIM), EMBEDDING_DIM, n_clusters
)
faiss_index.nprobe = 5  # Reduce search complexity

metadata_store = []

# Initialize tokenizer
logging.info("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load model
logging.info("Loading model...")
model = AutoModel.from_pretrained(MODEL_NAME).to("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

def chunk_text(text: str, chunk_size: int, overlap: int) -> list:
    """Splits text into properly sized chunks for embedding."""
    if not text.strip():  # Handle empty or whitespace-only strings
        return []

    tokens = tokenizer.encode(text)
    chunks = []
    for i in range(0, len(tokens), chunk_size - overlap):
        chunk = tokens[i:i + chunk_size]
        if chunk:
            chunks.append(tokenizer.decode(chunk))
    return chunks

# Embedding Generation Function
def generate_embeddings(texts, sectors):
    """Generate embeddings using average pooling with sector-based context."""
    if not texts:
        return []

    prefixed_texts = [
        f"Sector: {sector}. Given the summary and components of a development aid project, retrieve similar projects: {text}"
        for text, sector in zip(texts, sectors)
    ]

    batch_dict = tokenizer(prefixed_texts, max_length=400, padding=True, truncation=True, return_tensors="pt")
    batch_dict = {key: value.to(model.device) for key, value in batch_dict.items()}

    with torch.no_grad():
        outputs = model(**batch_dict)
        embeddings = average_pool(outputs.last_hidden_state, batch_dict["attention_mask"])
        embeddings = F.normalize(embeddings, p=2, dim=1)

    del batch_dict, outputs  # Free memory
    torch.cuda.empty_cache()
    gc.collect()
    return embeddings

def train_faiss_index(df, section_columns, num_samples=500):
    """Train FAISS using embeddings from the first few rows of the dataset."""
    logging.info("Training FAISS index with actual embeddings...")
    training_texts = []
    training_sectors = []
    for _, row in df.iterrows():
        for section in section_columns:
            if pd.notna(row[section]):
                training_texts.append(str(row[section]))
                training_sectors.append(row.get("Sector", "Other"))
            if len(training_texts) >= num_samples:
                break
        if len(training_texts) >= num_samples:
            break

    if training_texts:
        batch_size = 10
        collected_embeddings = []
        for i in range(0, len(training_texts), batch_size):
            batch = training_texts[i:i + batch_size]
            batch_sectors = training_sectors[i:i + batch_size]
            embeddings = generate_embeddings(batch, batch_sectors).cpu().numpy()
            collected_embeddings.append(embeddings)
            torch.cuda.empty_cache()
            gc.collect()

        training_embeddings = np.vstack(collected_embeddings)
        faiss_index.train(training_embeddings.astype(np.float32))
        logging.info("FAISS training completed!")
        del training_embeddings  # Free memory
        gc.collect()

def process_dataframe(df, project_type, id_column, section_columns):
    """Processes a DataFrame to generate embeddings and update the Faiss index."""
    global metadata_store
    logging.info(f"Starting processing for {project_type}, total rows: {len(df)}")
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Processing {project_type} data"):
        data_id = row[id_column]
        sector = row.get("Sector", "Other")
        for section in section_columns:
            if pd.notna(row[section]):
                text = str(row[section])
                chunks = chunk_text(text, CHUNK_SIZE, OVERLAP)
                for i in range(0, len(chunks), BATCH_SIZE):
                    batch = chunks[i:i + BATCH_SIZE]
                    embeddings = generate_embeddings(batch, [sector] * len(batch))
                    for chunk_idx, (embedding, chunk) in enumerate(zip(embeddings, batch)):
                        faiss_index.add(embedding.cpu().numpy().astype(np.float32).reshape(1, -1))
                        metadata_store.append({
                            "project_type": project_type,
                            "data_id": data_id,
                            "section": section,
                            "sector": sector,
                            "chunk_id": f"chunk_{idx}_{i + chunk_idx}",
                            "chunk_content": chunk
                        })
                    torch.cuda.empty_cache()
                    gc.collect()

logging.info("Processing complete. FAISS index and metadata saved.")

# 🚀 Train FAISS before processing
combined_df = pd.concat([afdb, aiib, world_bank])
train_faiss_index(combined_df, ["Project Components"], num_samples=500)

# Process the dataframes
process_dataframe(afdb, "AFDB", "Filename", afdb_columns)
process_dataframe(aiib, "AIIB", "Filename", aiib_columns)
process_dataframe(world_bank, "World Bank", "Project ID", world_bank_columns)

# Save FAISS index
faiss.write_index(faiss_index, "faiss_index_e5_V2.idx")

# Save metadata
with open("metadata_store_e5_V2.json", "w") as f:
    json.dump(metadata_store, f, indent=4)

logging.info("Processing complete. FAISS index and metadata saved.")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.18k [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Processing World Bank data: 100%|██████████| 2775/2775 [46:58<00:00,  1.02s/it]
